# TBBT - dobór odcinków

Wybiera odcinki wchodzące do korpusu na podstawie rankingu zapytań `v` z TVR i dzieli je na część deweloperską i testową. Niczego nie pobiera poza tytułami odcinków, gdy nie ma ich w buforze, i korzysta wyłącznie z biblioteki standardowej, więc działa niezależnie od stanu środowiska z PyTorch.

**Wymaga:** plików TVR (`tvr_train_release.jsonl`, `tvr_val_release.jsonl`) w `data/interim/tbbt`.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `data/interim/tbbt/tbbt_ranking_v_full.csv` | ranking wszystkich 220 odcinków wg liczby zapytań `v` |
| `data/interim/tbbt/tbbt_selection_<COUNT>.csv` | wybór odcinków z kolumną `split` (`dev`/`test`) |
| `data/interim/tbbt/tbbt_descriptions_v.csv` | opisy `v` wybranych odcinków, czasy jeszcze w skali klipu |

**Dalej:** `tbbt_02_annotation_candidates.ipynb`. Statystyki adnotacji liczy `tbbt_03_annotations.ipynb`, tam, gdzie powstają pliki, które je niosą.

In [ ]:

import csv
import json
import re
from collections import defaultdict
from pathlib import Path

ROOT     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
DATA_DIR = ROOT / "data" / "interim" / "tbbt"
FILES    = ["tvr_train_release.jsonl", "tvr_val_release.jsonl"]
# TBBT in TVR has no series prefix: vid_name = "s04e04_seg02_clip_12".
# The other series have a prefix (friends_, castle_, met_, grey_, house_).
PATTERN = re.compile(r"^(s\d{2}e\d{2})_seg")
RANKING_CSV = DATA_DIR / "tbbt_ranking_v_full.csv"

def build_ranking():
    """Returns (counter, ranking). counter: episode -> {v, vt, t, clips}.
    ranking: list of (episode, data) sorted descending by "v"."""
    counter = defaultdict(lambda: {"v": 0, "vt": 0, "t": 0, "clips": set()})
    for name in FILES:
        path = DATA_DIR / name
        if not path.exists():
            raise FileNotFoundError(
                f"Missing {path}. Put the TVR files (train+val) in {DATA_DIR}.")
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                m = PATTERN.match(rec.get("vid_name", ""))
                if not m:
                    continue  # a series other than TBBT (has a prefix)
                ep = m.group(1)
                kind = rec.get("type", "")
                if kind in ("v", "vt", "t"):
                    counter[ep][kind] += 1
                counter[ep]["clips"].add(rec["vid_name"])
    # descending by v; ties broken by episode id ascending (reproducibly)
    ranking = sorted(counter.items(), key=lambda kv: (-kv[1]["v"], kv[0]))
    return counter, ranking

## 1. Ranking wg zapytań `v`

**Zapisuje:** `data/interim/tbbt/tbbt_ranking_v_full.csv` - ranking wszystkich 220 odcinków, posortowany malejąco po liczbie zapytań `v`, bez kolumny `selected`.

Dla każdego odcinka zliczane są zapytania typu `v` (wyłącznie obraz), `vt` (obraz z napisami) i `t` (tylko napisy) oraz liczba klipów. Do ewaluacji wizualnej liczy się tylko `v`.

In [ ]:
counter, ranking = build_ranking()

DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(RANKING_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["episode", "queries_v", "queries_vt", "queries_t", "clip_count"])
    for ep, c in ranking:
        w.writerow([ep, c["v"], c["vt"], c["t"], len(c["clips"])])

print(f"TBBT episodes:    {len(ranking)}")
print(f"sum of v queries: {sum(c['v'] for _, c in ranking)}")
print(f"saved ranking (without a 'selected' column) -> {RANKING_CSV}")
print("\ntop 10 by v:")
for ep, c in ranking[:10]:
    print(f"  {ep}: v={c['v']}, vt={c['vt']}, t={c['t']}, clips={len(c['clips'])}")

## 2. Dobór odcinków i podział na dev/test

**Zapisuje:** `data/interim/tbbt/tbbt_selection_<COUNT>.csv` - wybrane odcinki z kolumną `split`.

Doborem sterują cztery zmienne: ile odcinków wziąć, ile z nich odłożyć na zbiór deweloperski, jaki minimalny odstęp zachować w obrębie sezonu i które sezony pominąć. Odcinki brane są round-robin po sezonach, a nie po kolei z rankingu, bo ranking `v` jest niemal płaski: TVR ma po 5 zapytań na klip, a odcinki mają podobną liczbę klipów.

Lista uporządkowana chronologicznie dzielona jest na `DEV_COUNT` równych bloków, a z każdego brany jest środek. Metoda jest deterministyczna, bez ziarna losowania, więc każde uruchomienie daje ten sam podział, a `dev` rozkłada się po całej rozpiętości serialu.

In [ ]:
#   EPISODE_COUNT  how many episodes to select in total
#   DEV_COUNT      how many of them to set aside for the dev split (the rest is test)
#   MIN_GAP        minimum gap between episode numbers WITHIN a season:
#                  MIN_GAP=3 => after s01e01 the next one from S1 is s01e04 at the earliest
#   SKIP_SEASONS   seasons excluded from the selection
EPISODE_COUNT = 24
DEV_COUNT = 6
MIN_GAP = 3
SKIP_SEASONS = ["s10"]     # S10 drifts from the TVQA source by ~3 s at scene changes

# Load the ranking from cell 1 (already sorted descending by v).
data = {}
rank_order = []
with open(RANKING_CSV, encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f, delimiter=";"):
        ep = r["episode"]
        if ep[:3] in SKIP_SEASONS:
            continue
        data[ep] = {"v": int(r["queries_v"]), "vt": int(r["queries_vt"]),
                    "t": int(r["queries_t"]), "clips": int(r["clip_count"])}
        rank_order.append(ep)

# Episodes of every season in ranking order (descending by v).
by_season = defaultdict(list)
for ep in rank_order:
    by_season[ep[:3]].append(int(ep[4:6]))
seasons = sorted(by_season)

selected = []
selected_in_season = defaultdict(list)

def available(season):
    """The highest-ranked episode of the season that keeps MIN_GAP from all
    episodes already selected in that season. None when there is none."""
    for no in by_season[season]:                 # order = ranking descending by v
        if no in selected_in_season[season]:
            continue
        if all(abs(no - p) >= MIN_GAP for p in selected_in_season[season]):
            return no
    return None

# Round by round: one episode per season by ranking; after the last season we
# return to the first. This way the seasons are represented evenly.
while len(selected) < EPISODE_COUNT:
    added = 0
    for season in seasons:
        if len(selected) >= EPISODE_COUNT:
            break
        no = available(season)
        if no is None:
            continue
        selected_in_season[season].append(no)
        selected.append(f"{season}e{no:02d}")
        added += 1
    if added == 0:
        print(f"WARNING: with MIN_GAP={MIN_GAP} it is not possible to select "
              f"{EPISODE_COUNT} episodes. Selected {len(selected)}.")
        break

# dev: middle of each of DEV_COUNT equal blocks of the chronological selection
chronological = sorted(selected)
step = len(chronological) / DEV_COUNT
dev = {chronological[int((i + 0.5) * step)] for i in range(DEV_COUNT)}

SELECTION_CSV = DATA_DIR / f"tbbt_selection_{EPISODE_COUNT}.csv"
with open(SELECTION_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["episode", "season", "episode_no", "split", "queries_v",
                "queries_vt", "queries_t", "clip_count"])
    for ep in chronological:
        c = data[ep]
        w.writerow([ep, ep[:3], int(ep[4:6]), "dev" if ep in dev else "test",
                    c["v"], c["vt"], c["t"], c["clips"]])

def summarize(label, items):
    v = sum(data[e]["v"] for e in items)
    cl = sum(data[e]["clips"] for e in items)
    sea = sorted({e[:3] for e in items})
    print(f"{label:<6}{len(items):>4} episodes{v:>7} v queries{cl:>6} clips   "
          f"seasons: {', '.join(s[1:] for s in sea)}")

print(f"selected {len(selected)} episodes from seasons {', '.join(s[1:] for s in seasons)}"
      + (f" (skipped {', '.join(SKIP_SEASONS)})" if SKIP_SEASONS else ""))
print(f"per season: { {s: len(v) for s, v in sorted(selected_in_season.items())} }\n")
summarize("total", chronological)
summarize("dev", [e for e in chronological if e in dev])
summarize("test", [e for e in chronological if e not in dev])

print(f"\n{'episode':<9}{'split':<7}{'v':>5}{'vt':>5}{'t':>5}{'clips':>8}")
for ep in chronological:
    c = data[ep]
    print(f"{ep:<9}{'dev' if ep in dev else 'test':<7}"
          f"{c['v']:>5}{c['vt']:>5}{c['t']:>5}{c['clips']:>8}")
print(f"\nsaved -> {SELECTION_CSV}")

## 3. Tytuły odcinków

Tytuły brane są z `work/tbbt_episode_titles.csv`, a gdy tego pliku nie ma - pobierane z TVmaze do pamięci.

In [ ]:
import urllib.request

EPISODE_COUNT = 24
SELECTION_CSV = DATA_DIR / f"tbbt_selection_{EPISODE_COUNT}.csv"
TITLES_CSV    = DATA_DIR / "work" / "tbbt_episode_titles.csv"
TVMAZE = ("https://api.tvmaze.com/singlesearch/shows"
          "?q=the%20big%20bang%20theory&embed=episodes")

# titles: from the cache if present, otherwise from TVmaze in memory (no saving)
title = {}
if TITLES_CSV.exists():
    with open(TITLES_CSV, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f, delimiter=";"):
            title[(r["season"], int(r["episode_no"]))] = r["title"]
else:
    with urllib.request.urlopen(TVMAZE, timeout=30) as r:
        data = json.load(r)
    for ep in data["_embedded"]["episodes"]:
        s, n, t = ep.get("season"), ep.get("number"), ep.get("name")
        if s and n:
            title[(f"s{int(s):02d}", int(n))] = t or ""

# load the selection, sort ascending by (season, episode) and show with the title
with open(SELECTION_CSV, encoding="utf-8-sig", newline="") as f:
    selection = list(csv.DictReader(f, delimiter=";"))
selection.sort(key=lambda r: (r["season"], int(r["episode_no"])))

print(f"{'episode':<10}{'split':<7}title")
print("-" * 52)
for r in selection:
    print(f"{r['episode']:<10}{r['split']:<7}"
          f"{title.get((r['season'], int(r['episode_no'])), '')}")

## 4. Opisy `v` dla wybranych odcinków

**Zapisuje:** `data/interim/tbbt/tbbt_descriptions_v.csv` - wszystkie zapytania typu `v` dla odcinków z wyboru, posortowane po `vid_name`. Kolumna `split` niesie podział na `dev`/`test`, żeby dalsze kroki nie musiały zaglądać do pliku z wyborem.

Znaczniki czasu (`ts_start`, `ts_end`) są tu jeszcze w skali klipu, liczone od jego początku. Na skalę całego odcinka przelicza je `tbbt_02_annotation_candidates`, który dokłada przesunięcie każdego klipu.

In [ ]:
EPISODE_COUNT = 24

SELECTION_CSV    = DATA_DIR / f"tbbt_selection_{EPISODE_COUNT}.csv"
DESCRIPTIONS_CSV = DATA_DIR / "tbbt_descriptions_v.csv"
PATTERN = re.compile(r"^(s\d{2}e\d{2})_(seg\d+)_clip_(\d+)$")

with open(SELECTION_CSV, encoding="utf-8-sig", newline="") as f:
    split = {r["episode"]: r["split"] for r in csv.DictReader(f, delimiter=";")}
selected = set(split)
print(f"episodes in the selection: {len(selected)} "
      f"(dev: {sum(1 for s in split.values() if s == 'dev')}, "
      f"test: {sum(1 for s in split.values() if s == 'test')})")

rows = []
for name in FILES:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the TVR files (train+val) in {DATA_DIR}.")
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if rec.get("type") != "v":          # visual queries only
                continue
            m = PATTERN.match(rec.get("vid_name", ""))
            if not m or m.group(1) not in selected:
                continue
            ts = rec["ts"]
            rows.append({
                "desc_id": rec["desc_id"],
                "vid_name": rec["vid_name"],
                "episode": m.group(1),
                "split": split[m.group(1)],
                "segment": m.group(2),
                "clip_no": int(m.group(3)),
                "ts_start": round(float(ts[0]), 3),
                "ts_end": round(float(ts[1]), 3),
                "clip_duration": round(float(rec["duration"]), 3),
                "desc": " ".join(rec["desc"].split()),   # whitespace normalization only
            })

# vid_name split into components, so that clip_2 does not land after clip_10
rows.sort(key=lambda r: (r["episode"], r["segment"], r["clip_no"], r["ts_start"], r["desc_id"]))

DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(DESCRIPTIONS_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter=";")
    w.writeheader()
    w.writerows(rows)

clips = {r["vid_name"] for r in rows}
lengths = [r["ts_end"] - r["ts_start"] for r in rows]
print(f"v descriptions:    {len(rows)}")
print(f"distinct clips:    {len(clips)}")
print(f"fragment length:   median {sorted(lengths)[len(lengths)//2]:.1f} s, "
      f"min {min(lengths):.1f} s, max {max(lengths):.1f} s")
print(f"saved -> {DESCRIPTIONS_CSV}\n")

for s in ("dev", "test"):
    w = [r for r in rows if r["split"] == s]
    print(f"{s:<5}{len({r['episode'] for r in w}):>3} episodes, {len(w):>4} descriptions, "
          f"{len({r['vid_name'] for r in w}):>3} clips")

print("\ndescriptions per episode:")
for ep in sorted(selected):
    n = sum(1 for r in rows if r["episode"] == ep)
    k = len({r["vid_name"] for r in rows if r["episode"] == ep})
    print(f"  {ep} [{split[ep]:<4}]: {n:>3} descriptions in {k:>2} clips")

print("\nfirst 5 rows:")
for r in rows[:5]:
    print(f"  {r['vid_name']:<24}{r['ts_start']:>7.2f}-{r['ts_end']:<8.2f}{r['desc']}")